In [1]:
import pandas as pd
import numpy as np
import json
import src.utils.iv_helpers as iv_h
import src.utils.feature_eng as feat_eng
from src.utils.demeaning import demean_2FE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from sklearn.model_selection import GroupKFold
from pathlib import Path

# Loading Data
We load the panel and construct demeaned outcome and instrument matrix

In [2]:
IN = Path("../data/processed")

final = json.loads((IN / "final.json").read_text())
df = pd.read_parquet(IN / "bfs_data.parquet")

y = final["endog"]
Z = final["Z_selected"]

# Rebuilding multi-index
dfp = df.set_index([final["entity_level"], final["time_level"]]).sort_index()

# Demeaning using helper function
df_dm = demean_2FE(dfp[[y] + Z].dropna())
# Outcome vector
y_dm = df_dm[y]
# Regressor matrix
X_dm = df_dm[Z]

# Scaling for LASSO
Lasso is sensitive to scale so we standardize the instrument columns.

In [3]:
scaler = StandardScaler(with_mean=True, with_std=True)

X_std = scaler.fit_transform(X_dm.values)
dfX_std = pd.DataFrame(X_std, index=X_dm.index, columns=X_dm.columns)

# Checking if standardization worked (mean=0, std=1)
print(dfX_std.describe().loc[["mean","std"]].round().head())

      Z_temp_m2  Z_temp_m4  Z_temp_m5  Z_temp_m6  Z_temp_m7  Z_temp_m8  \
mean        0.0       -0.0       -0.0       -0.0       -0.0       -0.0   
std         1.0        1.0        1.0        1.0        1.0        1.0   

      Z_prcp_m1  Z_prcp_m5  Z_prcp_m6  Z_prcp_m8  Z_prcp_m11  Z_prcp_m12  \
mean        0.0       -0.0       -0.0       -0.0         0.0         0.0   
std         1.0        1.0        1.0        1.0         1.0         1.0   

      Z_temp_m12  Z_prcp_m3  
mean        -0.0       -0.0  
std          1.0        1.0  


# Feature Engineering
We will transform the base instruments according to the following rules:

### Polynomials
+ $x = x^2$
+ $x = x^3$

### Absolute Value
+ $x = abs(x)$

### Hinges
+ $x = max[0, x-c_q]$
+ $x = max[0, c_q-x]$

In [4]:
X_temp = dfX_std.copy()

X_nonlinear = feat_eng.build_non_linear_feats(X_temp)

print(len(X_nonlinear.columns))
print(X_nonlinear.columns)

140
Index(['Z_temp_m2', 'Z_temp_m4', 'Z_temp_m5', 'Z_temp_m6', 'Z_temp_m7',
       'Z_temp_m8', 'Z_prcp_m1', 'Z_prcp_m5', 'Z_prcp_m6', 'Z_prcp_m8',
       ...
       'Z_temp_m12__hinge_pos_q50', 'Z_temp_m12__hinge_neg_q50',
       'Z_temp_m12__hinge_pos_q75', 'Z_temp_m12__hinge_neg_q75',
       'Z_prcp_m3__hinge_pos_q25', 'Z_prcp_m3__hinge_neg_q25',
       'Z_prcp_m3__hinge_pos_q50', 'Z_prcp_m3__hinge_neg_q50',
       'Z_prcp_m3__hinge_pos_q75', 'Z_prcp_m3__hinge_neg_q75'],
      dtype='str', length=140)


# Running LASSO Group K-fold
We choose the LASSO penalty α based on cross-validation that respects panel clustering.

In [5]:
X_vals = X_nonlinear.values
y_vals = y_dm.values

groups = X_nonlinear.index.get_level_values("entity_id").to_numpy()

gkf = GroupKFold(n_splits=5)

lasso_cv = LassoCV(
    cv=gkf.split(X_vals, y_vals, groups=groups),
    alphas=100,
    max_iter=20000,
    random_state=0,
)

lasso_cv.fit(X_vals, y_vals)

print("Chosen alpha:", lasso_cv.alpha_)
print("Nonzero coefs:", np.sum(lasso_cv.coef_ != 0))

Chosen alpha: 0.000335531756583379
Nonzero coefs: 59


# Testing Over Subsamples For Stability
For robustness, we identify the instruments consistently picked across subsamples.

In [6]:
X_all = X_nonlinear.values
y_all = y_dm.values
feature_names = X_nonlinear.columns.to_numpy()

# Extracting alpha
alpha = float(lasso_cv.alpha_)

# Resampling entities
entities = X_nonlinear.index.get_level_values("entity_id").to_numpy()
unique_entities = np.unique(entities)

# No. sample runs
B = 200
# Fraction of entities per run
frac = 0.7
# Seeding
rng = np.random.default_rng(0)

sel_counts = np.zeros(X_all.shape[1], dtype=int)
coef_sums = np.zeros(X_all.shape[1], dtype=float)

for b in range(B):
    m = int(np.ceil(frac * len(unique_entities)))
    # Random sampling of entities
    sampled_ents = rng.choice(unique_entities, size=m, replace=False)

    mask = np.isin(entities, sampled_ents)
    X_b = X_all[mask]
    y_b = y_all[mask]

    # Fitting LASSO at fixed alpha
    model = Lasso(alpha=alpha, max_iter=20000, random_state=0)
    model.fit(X_b, y_b)

    nz = model.coef_ != 0
    sel_counts[nz] += 1
    coef_sums[nz] += model.coef_[nz]

# Selection frequencies + mean coef conditional on selection
sel_freq = sel_counts / B
mean_coef = np.zeros_like(coef_sums)
nonzero_mask = sel_counts > 0
mean_coef[nonzero_mask] = coef_sums[nonzero_mask] / sel_counts[nonzero_mask]

stable_cand = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "sel_freq": sel_freq,
            "mean_coef_if_selected": mean_coef,
        }
    )
    .sort_values(["sel_freq", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

stable_cand.head()

,feature,sel_freq,mean_coef_if_selected
0,Z_prcp_m5__cu,1.0,-0.002889
1,Z_prcp_m8__cu,1.0,-0.000648
2,Z_temp_m2__cu,1.0,-0.001829
3,Z_temp_m6__cu,1.0,-0.002482
4,Z_temp_m7__cu,1.0,-0.001617


# Imposing Threshold On Stable Set
We keep the features that were picked above a certain threshold.

In [7]:
# Descriptive stats of candidate set
stable_cand.describe()

# Choosing median 0.7
THRESH = 0.7

# Filtering
stable_cand = stable_cand[stable_cand["sel_freq"] > THRESH]

print(stable_cand.shape)

(44, 3)


# Pruning stable set
Since too many variables remain, we prune the instrument set by imposing the following rule:

+ For a base feature, prefer its hinge transformation
+ Otherwise choose up to two of linear, cubic and square - but not square and cubic

In [8]:
stable = stable_cand.copy()
stable['feat_name'] = stable['feature'].apply(iv_h.feat_name)
stable['feat_trans'] = stable['feature'].apply(iv_h.feat_transformation)
stable['hinge_q'] = stable['feature'].apply(iv_h.hinge_quartile_search)

In [9]:
# Imposing stricter threshold
pooled = stable[stable["sel_freq"] >= 0.90].copy()

# Grouping by feature name
grouped = pooled.groupby('feat_name')

chosen_vars = []

for name, group in grouped:
    group = group.sort_values('sel_freq', ascending=False)

    # Hinge takes priority
    hinges = group[group['feat_trans'] == "hinge"]
    # Checking if hinges exist
    if len(hinges):
        # Choosing the best hinge
        chosen_vars.append(hinges.iloc[0]['feature'])
        continue

    # Otherwise check for other transformations and prefer linear/sq
    cand = group[group['feat_trans'].isin(['linear','square', 'cubic'])].copy()
    if len(cand):
        chosen = []
        for f in cand['feature']:
            # If 2 were chosen break the loop
            if len(chosen) >= 2: break
            # Avoid selecting both square and cubic
            if f.endswith('__cu') and any(x.endswith('__sq') for x in chosen):
                continue
            chosen.append(f)
        chosen_vars += chosen

Z_new = chosen_vars
print(Z_new, len(Z_new))

['Z_prcp_m1__hinge_neg_q25', 'Z_prcp_m11__cu', 'Z_prcp_m12__hinge_pos_q25', 'Z_prcp_m3__cu', 'Z_prcp_m5__cu', 'Z_prcp_m5__sq', 'Z_prcp_m6__cu', 'Z_prcp_m8__cu', 'Z_prcp_m8__sq', 'Z_temp_m12__cu', 'Z_temp_m2__hinge_pos_q25', 'Z_temp_m4__hinge_pos_q25', 'Z_temp_m5__cu', 'Z_temp_m6__hinge_neg_q75', 'Z_temp_m7__cu', 'Z_temp_m7__sq', 'Z_temp_m8__cu'] 17


# Running FE on pruned set
We re-run the FE regression on the new pruned set of instruments. We then run a Wald test to verify the instruments have joint significance i.e. they have explanatory power.

In [10]:
# Creating new df with transformations
Z = Z_new
y = 'log_yield'

# Concatenating dfs
df_fe = pd.concat(
    [dfp[[y]], X_nonlinear[Z]],
    axis=1
).dropna()

model = PanelOLS(df_fe[y],
                 df_fe[Z],
                 entity_effects=True,
                 time_effects=True
                 ).fit(cov_type='clustered', cluster_entity=True)


# joint test that all Z coefficients are zero
wald = model.wald_test(formula=[f"{z} = 0" for z in Z])
print(wald)

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 274.4907
P-value: 0.0000
Distributed: chi2(17)


# Interpretation
We strongly reject null that instruments dont have jointly significant explanatory power at 0.01% confidence level.


# 2SLS
We run a two-stage least squares regression with the new instrument set to estimate its causal effect on area harvested in the following year. We then run an overidentification test.

In [11]:
# Saving list of instruments
Z_final = Z

X_nl = X_nonlinear.copy()

# Aligning index
dfp_aligned = dfp.loc[X_nl.index]

# Creating iv dataframe
df_iv = pd.concat([dfp_aligned, X_nl[Z_final]], axis=1).reset_index()

# Dropping duplicate cols
df_iv = df_iv.loc[:, ~df_iv.columns.duplicated()].copy()


In [12]:
Z = Z_final
endog = 'log_yield'
y = "log_area_lead1"

cols_needed = [y, endog] + Z

df_temp = df_iv.dropna(subset=cols_needed)

formula = f"{y} ~ 1 + [{endog} ~ {' + '.join(Z)}] + C(entity_id) + C(year)"

iv_res = IV2SLS.from_formula(
    formula,
    data=df_temp
).fit(cov_type="clustered", clusters=df_temp["entity_id"])

print(iv_res.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:         log_area_lead1   R-squared:                      0.9455
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9432
No. Observations:                2631   F-statistic:                   4.8e+19
Date:                Fri, May 15 2026   P-value (F-stat)                0.0000
Time:                        10:37:00   Distribution:                chi2(105)
Cov. Estimator:             clustered                                         
                                                                              
                                    Parameter Estimates                                     
                          Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------
Intercept                    10.768     0.6918     15.565     0.0000      9.4125      12.

# Diagnostics Tests
## Wooldridge Score Test Of Overidentification
We reject the null that the model is not overidentified at 0.0001 confidence level. There is significant evidence that the model is overidentified.

In [13]:
print(iv_res.wooldridge_overid, '\n')

Wooldridge's score test of overidentification
H0: Model is not overidentified.
Statistic: 44.8663
P-value: 0.0001
Distributed: chi2(16) 



# Model Overidentified
We need to just identify it and pick the instrument with strongest first stage, reasonable SE and interpretable coefficient.

In [14]:
y = "log_area_lead1"
endog = "log_yield"
Z = Z_final

cols = [y, endog] + Z
df = df_iv.set_index(["entity_id","year"])[cols].dropna()

# Manually demeaning to avoid creating dummies
df_dm = demean_2FE(df)
y_dm = df_dm[y]
x_dm = df_dm[endog]

results = []

for z in Z:
    z_dm = df_dm[z]
    # Fitting each instrument individually
    r = IV2SLS(y_dm, None, x_dm, z_dm).fit(cov_type="clustered", clusters=df.index.get_level_values("entity_id"))

    fs = r.first_stage
    # Diagnostics for first stage
    diag = fs.diagnostics

    # Appending regression results to dictionary
    results.append({"Z": z,
                    "beta_log_yield": round(r.params["log_yield"], 3),
                    "se": round(r.std_errors["log_yield"],3),
                    "first_stage_F": round(diag.loc[endog, "f.stat"],3),
                    "first_stage_p": round(diag.loc[endog, "f.pval"],3),
                    "partial_R2": round(diag.loc[endog, "partial.rsquared"],3)})

# Sorting by standard error
results_df = pd.DataFrame(results).sort_values("se")
results_df.head()

,Z,beta_log_yield,se,first_stage_F,first_stage_p,partial_R2
14,Z_temp_m7__cu,-0.693,0.419,34.991,0.000,0.014
13,Z_temp_m6__hinge_neg_q75,-1.039,0.559,23.371,0.000,0.014
16,Z_temp_m8__cu,-0.511,0.722,12.387,0.000,0.007
10,Z_temp_m2__hinge_pos_q25,0.905,0.755,8.675,0.003,0.004
12,Z_temp_m5__cu,0.935,0.832,11.245,0.001,0.004


# Running IV With Strongest Instruments
We choose the three instruments from our instrument set with the highest F score in the first stage. We then run 2SLS with each of the instruments and record the diagnostics.

In [15]:
# Sorting by first stage F
Z_list = results_df.sort_values("first_stage_F", ascending=False)["Z"].head(3).tolist()
y = "log_area_lead1"
endog = "log_yield"

cols = [y, endog, "entity_id", "year"] + Z_list
df0 = df_iv[cols].dropna().copy()

rows = []
for z in Z_list:
    formula = f"{y} ~ 1 + C(entity_id) + C(year) + [{endog} ~ {z}]"
    r = IV2SLS.from_formula(formula, data=df0).fit(
        cov_type="clustered",
        clusters=df0["entity_id"]
    )
    diag = r.first_stage.diagnostics
    rows.append({
        "Z": z,
        "N": int(r.nobs),
        "beta": r.params[endog],
        "se": r.std_errors[endog],
        "F": diag.loc[endog, "f.stat"],
        "pF": diag.loc[endog, "f.pval"],
        "partial_R2": diag.loc[endog, "partial.rsquared"],
    })

df_iv_res = pd.DataFrame(rows).sort_values("F", ascending=False)

df_iv_res

,Z,N,beta,se,F,pF,partial_R2
0,Z_temp_m7__cu,2631,-0.693206,0.418919,34.990886,3.312522e-09,0.013631
1,Z_temp_m6__hinge_neg_q75,2631,-1.039332,0.559082,23.371492,1.335431e-06,0.014486
2,Z_prcp_m5__cu,2631,1.568640,0.845719,16.163430,5.810513e-05,0.009465


# Testing Exclusion Restriction
We check how sensitive the instruments are by adding nearby weather controls which could capture direct effects e.g. Z_temp_m7__cu gets Z_temp_m8__x. We then re-estimate 2SLS.

In [16]:
temp_months = [c for c in df_iv.columns if c.startswith("Z_temp_m")]
prcp_months = [c for c in df_iv.columns if c.startswith("Z_prcp_m")]

In [17]:
# Instrument is a temp month 7 cubed
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_temp_m7__cu",
          controls=iv_h.controls_temp(7, temp_months=temp_months, prcp_months=prcp_months))

# Instrument is a temp month 6 cubed
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_temp_m6__hinge_neg_q75",
          controls=iv_h.controls_temp(6, temp_months=temp_months, prcp_months=prcp_months))

# Instrument is a temp month 6 squared
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_prcp_m5__cu",
          controls=iv_h.controls_prcp(5, temp_months=temp_months, prcp_months=prcp_months))


Exclusion restriction stress test for Z_temp_m7__cu
--------------------------------------
Baseline:  β = -0.693 (SE = 0.419)
Stress:    β = -0.464 (SE = 0.369)


Exclusion restriction stress test for Z_temp_m6__hinge_neg_q75
--------------------------------------
Baseline:  β = -1.039 (SE = 0.559)
Stress:    β = -0.820 (SE = 0.431)


Exclusion restriction stress test for Z_prcp_m5__cu
--------------------------------------
Baseline:  β = 1.569 (SE = 0.846)
Stress:    β = 1.255 (SE = 0.650)



# Final Regression
Our final 2SLS regression uses the strongest instrument -> $\text{Z\_temp\_m6\_\_hinge\_neg\_q75}$

In [18]:
y = "log_area_lead1"
endog = "log_yield"
z = "Z_temp_m6__hinge_neg_q75"

cols = [y, endog, z, "entity_id", "year"]
df_use = df_iv[cols].dropna().copy()

formula = f"{y} ~ 1 + C(entity_id) + C(year) + [{endog} ~ {z}]"
iv_final = IV2SLS.from_formula(formula, data=df_use).fit(
    cov_type="clustered",
    clusters=df_use["entity_id"]
)

print("beta:", iv_final.params[endog])
print("se:", iv_final.std_errors[endog])
print("p:", iv_final.pvalues[endog])
print(iv_final.first_stage.diagnostics)


beta: -1.0393320414004847
se: 0.5590824204250542
p: 0.06302769061671731
           rsquared  partial.rsquared  shea.rsquared     f.stat    f.pval  \
log_yield  0.840012          0.014486       0.014486  23.371492  0.000001   

            f.dist  
log_yield  chi2(1)  


# Double-LASSO IV
We partial out the remaining weather controls from outcome, treatment, and instrument
separately using LASSO, then run IV on the three sets of residuals. This ensures the
standard errors account for the uncertainty introduced by the earlier instrument selection step.

In [21]:
# Exclude full family of the base instrument
base_instrument = iv_h.feat_name(z)
W_cols = [c for c in X_nonlinear.columns if iv_h.feat_name(c) != base_instrument]

# Align all columns onto a common index and drop incomplete rows
df_dl = (
    pd.concat([dfp_aligned[["log_area_lead1", endog]], X_nonlinear], axis=1)
    [["log_area_lead1", endog, z] + W_cols]
    .dropna()
)

# Two-way demean
df_dm_dl = demean_2FE(df_dl)

Y_dm = df_dm_dl["log_area_lead1"].values
D_dm = df_dm_dl[endog].values
Z_dm = df_dm_dl[z].values
W_dm = df_dm_dl[W_cols].values

# Entity labels for clustering and grouped CV
entity_idx = df_dm_dl.index.get_level_values("entity_id").to_numpy()
cv_splits  = list(GroupKFold(n_splits=5).split(W_dm, Y_dm, entity_idx))

# Scale W (LASSO is scale-sensitive)
W_std = StandardScaler().fit_transform(W_dm)

# 1. Partial W out of Y
lasso_y = LassoCV(cv=cv_splits, fit_intercept=False, max_iter=50000, n_jobs=-1)
lasso_y.fit(W_std, Y_dm)
y_tilde = Y_dm - lasso_y.predict(W_std)
print(f"Y ~ W : α={lasso_y.alpha_:.5f}, nonzero={np.sum(lasso_y.coef_ != 0)}")

# 2. partial W out of D
lasso_d = LassoCV(cv=cv_splits, fit_intercept=False, max_iter=50000, n_jobs=-1)
lasso_d.fit(W_std, D_dm)
d_tilde = D_dm - lasso_d.predict(W_std)
print(f"D ~ W : α={lasso_d.alpha_:.5f}, nonzero={np.sum(lasso_d.coef_ != 0)}")

# 3. partial W out of Z
lasso_z = LassoCV(cv=cv_splits, fit_intercept=False, max_iter=50000, n_jobs=-1)
lasso_z.fit(W_std, Z_dm)
z_tilde = Z_dm - lasso_z.predict(W_std)
print(f"Z ~ W : α={lasso_z.alpha_:.5f}, nonzero={np.sum(lasso_z.coef_ != 0)}")

# 4. IV on residuals
df_resid = pd.DataFrame({
    "y_tilde": y_tilde,
    "d_tilde": d_tilde,
    "z_tilde": z_tilde,
    "entity":  entity_idx,
})

res_dl = IV2SLS(
    dependent   = df_resid["y_tilde"],
    exog        = None,
    endog       = df_resid[["d_tilde"]],
    instruments = df_resid[["z_tilde"]],
).fit(cov_type="clustered", clusters=df_resid["entity"])

# Building comparison table
naive_F = iv_final.first_stage.diagnostics.loc[endog,     "f.stat"]
dl_F    = res_dl.first_stage.diagnostics.loc["d_tilde", "f.stat"]

comparison = pd.DataFrame({
    "Estimator": ["Naive post-LASSO IV", "Double-LASSO IV"],
    "β":  [round(iv_final.params[endog], 4), round(res_dl.params["d_tilde"], 4)],
    "SE": [round(iv_final.std_errors[endog], 4), round(res_dl.std_errors["d_tilde"], 4)],
    "p":  [round(iv_final.pvalues[endog], 4), round(res_dl.pvalues["d_tilde"], 4)],
    "First-stage F": [round(naive_F, 3), round(dl_F, 3)],
})
print(comparison.to_string(index=False))

/home/adrym/projects/iv_weather_yields_selection/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.034e-02, tolerance: 3.389e-02
  model = cd_fast.enet_coordinate_descent_gram(
/home/adrym/projects/iv_weather_yields_selection/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:701: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.207e-02, tolerance: 3.389e-02
  model = cd_fast.enet_coordinate_descent_gram(


Y ~ W : α=0.00797, nonzero=27
D ~ W : α=0.00155, nonzero=36
Z ~ W : α=0.01185, nonzero=27
          Estimator       β     SE      p  First-stage F
Naive post-LASSO IV -1.0393 0.5591 0.0630         23.371
    Double-LASSO IV -0.8649 0.4464 0.0527         46.527


# Conclusion

## Second Stage
+ $\beta_{\text{naive}} = -1.039$ (SE = 0.559, $p = 0.063$)
+ $\beta_{\text{DL}} = -0.865$ (SE = 0.446, $p = 0.053$)

The preferred estimate follows Belloni, Chernozhukov & Hansen (2012): we partial out
the remaining nonlinear weather controls from outcome, treatment, and instrument
separately via LASSO before running IV on the residuals, correcting for the
post-selection inference problem introduced by the earlier instrument search.

A 1% increase in current-year yield leads to a **0.86% decrease** in area harvested
the following year, conditional on entity and year fixed effects. The direction and
magnitude are stable across both specifications. This is consistent with a standard
cobweb supply adjustment: farmers reduce planting after a positive yield shock,
anticipating lower prices.

The result is marginally significant ($p = 0.053$), falling just outside the 5%
threshold under the preferred specification. Evidence is present but not strong.

## First Stage
+ $\text{Partial } R^2 = 0.0145$
+ $p < 0.0001$
+ $\text{F-stat}_{\text{naive}} = 23.37$, $\text{F-stat}_{\text{DL}} = 46.53$

Both specifications exceed the conventional threshold of 10 for instrument strength.
The double LASSO first-stage F is substantially higher because the instrument family
exclusion ensures the residual instrument variation is genuinely independent of the
remaining weather controls, rather than partially absorbed by them.

## Exclusion Restriction
Stress-testing the chosen instrument by adding adjacent-month weather controls
produces moderate coefficient attenuation ($\beta: -1.039 \to -0.820$) but no sign
reversal and no collapse in significance, consistent with the exclusion restriction
holding approximately.

## Economic Interpretation
The instrument captures **extreme negative June temperature shocks**, deviations
below the 75th-percentile threshold of the state-specific historical distribution.
June falls within the critical vegetative growth window for corn, making severe cold
shocks a sharp, localised source of yield damage that is plausibly exogenous to
farmers' forward planting decisions except through the yield outcome it causes.

## Analysis Conclusions
Over a broad set of linear and nonlinear weather instruments, June temperature shocks
emerged endogenously as the most relevant and credible source of identifying variation.
This suggests mid-season yield signals play the dominant role in farmers' subsequent
planting decisions, consistent with adaptive expectations updating on observed outcomes.